# 🚀 Sanity Check & Optimized Full Training YOLOv8 on RTX 3050 Ti (4GB VRAM)

Notebook này chứa quy trình kiểm tra môi trường, nạp thử nghiệm và huấn luyện tối ưu hóa trên GPU RTX 3050 Ti (4GB VRAM):
1. **Kiểm tra CUDA & Hướng dẫn cài đặt**: Xác định xem PyTorch đã nhận GPU CUDA chưa và cách xử lý lỗi GPU chạy 2%.
2. **Script 1: Trích xuất Dataset (Dataset Slicer)**: Trích xuất 5% dữ liệu làm tập thử nghiệm.
3. **Script 2: Huấn luyện Thử nghiệm (Trial Training)**: Sanity check 20 epochs trên 5% dữ liệu.
4. **Script 3: Huấn luyện Toàn bộ Dữ liệu (Full Training)**: Huấn luyện trên toàn bộ ~12,500 ảnh với cấu hình tối ưu hóa tài nguyên GPU.

## 🩺 1. Kiểm tra CUDA & GPU (CUDA & GPU Diagnostics)

Nếu GPU chỉ chạy **2%**, nguyên nhân lớn nhất là PyTorch đang chạy bằng **CPU** (do cài đặt bản PyTorch không hỗ trợ CUDA) hoặc bạn đang xem biểu đồ **3D** thay vì **CUDA** trên Windows Task Manager.

Chạy ô code dưới đây để kiểm tra trạng thái thực tế:

In [5]:
import torch
import sys

print(f"🐍 Phiên bản Python: {sys.version}")
print(f"🔥 Phiên bản PyTorch: {torch.__version__}")
print(f"🤖 CUDA có khả dụng (GPU): {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"⚙️ Phiên bản CUDA của PyTorch: {torch.version.cuda}")
else:
    print("❌ CẢNH BÁO: PyTorch chưa nhận diện được GPU CUDA! Quá trình huấn luyện đang chạy trên CPU (rất chậm).")

🐍 Phiên bản Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
🔥 Phiên bản PyTorch: 2.5.1+cu121
🤖 CUDA có khả dụng (GPU): True
✅ Tên GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
⚙️ Phiên bản CUDA của PyTorch: 12.1


### 📦 Hướng Dẫn Cài Đặt Lại PyTorch CUDA (Nếu báo `False` ở trên)

Nếu hệ thống báo `CUDA có khả dụng (GPU): False`, bạn cần gỡ cài đặt bản torch hiện tại và cài đặt lại phiên bản hỗ trợ GPU tương thích với CUDA 12.1 hoặc 11.8:

```bash
# 1. Kích hoạt môi trường ảo (Virtual Environment) của dự án
# 2. Gỡ cài đặt bản PyTorch chỉ chạy CPU hiện tại
pip uninstall -y torch torchvision torchaudio

# 3. Cài đặt lại bản PyTorch hỗ trợ GPU CUDA 12.1
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 4. Cài đặt thư viện Ultralytics và các thư viện hỗ trợ
pip install ultralytics tqdm pandas
```

> **💡 Mẹo nhỏ xem Task Manager trên Windows:**
> Khi huấn luyện, bạn mở **Task Manager** -> Chọn tab **Performance** -> Chọn card **NVIDIA GPU** -> Nhấn vào tiêu đề của một trong các biểu đồ nhỏ (ví dụ chọn thay cho *Copy* hoặc *3D*) và đổi thành **CUDA** hoặc **Compute_0**. Lúc này bạn sẽ thấy GPU thật sự hoạt động ở mức cao (>80%).

## ✂️ 2. Script 1: Trích xuất Dataset (Dataset Slicer)

Đoạn mã dưới đây sẽ trích xuất ngẫu nhiên **đúng 5% dữ liệu** từ tập `combined_pothole` để tạo ra tập dữ liệu nhỏ phục vụ mục đích huấn luyện thử nghiệm (`trial_dataset`), đồng thời tự động sinh file `dataset.yaml` chuẩn YOLO.

In [11]:
!nvidia-smi -l 1

^C
Wed Jun 24 00:41:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.83                 Driver Version: 576.83         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P8              3W /   40W |    1505MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+--

In [6]:
import os
import random
import shutil
from pathlib import Path

# Cấu hình đường dẫn dự án
import os
# Tự động xác định PROJECT_ROOT
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'combined_pothole'
TRIAL_DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'trial_dataset'

def slice_dataset(src_dir, dest_dir, slice_ratio=0.05):
    splits = ['train', 'val']
    
    # Dọn dẹp thư mục cũ nếu tồn tại để tránh ghi đè rác dữ liệu
    if dest_dir.exists():
        print(f"🧹 Đang dọn dẹp thư mục trial cũ tại: {dest_dir}")
        shutil.rmtree(dest_dir)
        
    for split in splits:
        src_img_dir = src_dir / split / 'images'
        src_lbl_dir = src_dir / split / 'labels'
        
        dest_img_dir = dest_dir / split / 'images'
        dest_lbl_dir = dest_dir / split / 'labels'
        
        dest_img_dir.mkdir(parents=True, exist_ok=True)
        dest_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        # Quét tất cả các file ảnh
        img_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
        img_files = []
        for ext in img_extensions:
            img_files.extend(list(src_img_dir.glob(f'*{ext}')))
            
        # Lọc trích xuất ngẫu nhiên đúng 5% với seed cố định
        random.seed(42)
        num_slice = max(1, int(len(img_files) * slice_ratio))
        selected_imgs = random.sample(img_files, num_slice)
        
        print(f"📦 Split [{split}]: Đã chọn ngẫu nhiên {len(selected_imgs):,} / {len(img_files):,} ảnh ({slice_ratio * 100}%)")
        
        # Tiến hành copy ảnh và nhãn
        for img_path in selected_imgs:
            shutil.copy2(img_path, dest_img_dir / img_path.name)
            
            lbl_path = src_lbl_dir / f"{img_path.stem}.txt"
            dest_lbl_path = dest_lbl_dir / f"{img_path.stem}.txt"
            if lbl_path.exists():
                shutil.copy2(lbl_path, dest_lbl_path)
            else:
                with open(dest_lbl_path, 'w') as f:
                    pass
                    
    # Tự động khởi tạo file dataset.yaml chuẩn YOLOv8
    yaml_content = f"""path: {dest_dir.as_posix()}
train: train/images
val: val/images

names:
  0: Pothole
"""
    yaml_path = dest_dir / 'dataset.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(yaml_content)
        
    print(f"\n🎉 Đã hoàn thành tạo dữ liệu mẫu và ghi cấu hình tại: {yaml_path}")

# Chạy hàm trích xuất
slice_dataset(SRC_DATASET_DIR, TRIAL_DATASET_DIR, slice_ratio=0.05)

🧹 Đang dọn dẹp thư mục trial cũ tại: d:\Research\Yolo_Pothole_detection\Pothole_Detection\data\processed\trial_dataset
📦 Split [train]: Đã chọn ngẫu nhiên 1,253 / 25,078 ảnh (5.0%)
📦 Split [val]: Đã chọn ngẫu nhiên 253 / 5,062 ảnh (5.0%)

🎉 Đã hoàn thành tạo dữ liệu mẫu và ghi cấu hình tại: d:\Research\Yolo_Pothole_detection\Pothole_Detection\data\processed\trial_dataset\dataset.yaml


## 🚀 3. Script 2: Huấn Luyện Thử Nghiệm (YOLOv8 Trial Training)

Mã nguồn dưới đây thực hiện huấn luyện thử nghiệm YOLOv8s trên **tập mẫu 5%** (`trial_dataset`) để kiểm tra tính đúng đắn của đường dẫn nhãn và xem hướng đi xuống của Loss trong 20 epochs.

In [8]:
from ultralytics import YOLO
from pathlib import Path

# Cấu hình đường dẫn
import os
# Tự động xác định PROJECT_ROOT
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
YAML_PATH = PROJECT_ROOT / 'data' / 'processed' / 'trial_dataset' / 'dataset.yaml'

# Khởi tạo mô hình YOLOv8s từ weights của hãng
print("📥 Đang tải mô hình pre-trained YOLOv8s...")
model = YOLO('yolov8s.pt')

print("🔥 Bắt đầu chạy sanity check / trial training...")
results = model.train(
    data=str(YAML_PATH),
    epochs=20,             # Chạy 20 epochs để kiểm tra tính hội tụ nhanh và không lỗi
    
    # 1. 🛡️ CẤU HÌNH TRÁNH LỖI CUDA OUT OF MEMORY (RTX 3050 Ti - 4GB VRAM)
    imgsz=512,             # Giảm kích thước ảnh đầu vào xuống 512x512 để tiết kiệm bộ nhớ VRAM
    batch=4,               # Batch size cực nhỏ để giữ lượng activation maps thấp trong VRAM
    device=0,              # Chỉ định huấn luyện trên GPU CUDA:0
    workers=2,             # Số lượng luồng nạp dữ liệu nhỏ giúp ổn định bộ nhớ CPU/GPU
    amp=True,              # Bật FP16 Mixed Precision giúp giảm gần một nửa lượng VRAM
    cache=False,           # Tắt cache tránh tràn bộ nhớ đột ngột
    
    # 2. 📈 CẤU HÌNH ỔN ĐỊNH GRADIENT KHI BATCH SIZE NHỎ
    optimizer='AdamW',     # AdamW hoạt động ổn định và hội tụ tốt khi batch size nhỏ
    lr0=0.001,             # Học suất thấp để hạn chế gradient dao động quá lớn
    cos_lr=True,           # Lịch trình giảm học suất Cosine Annealing mượt mà
    warmup_epochs=3,       # Khởi động ấm 3 epoch giúp ổn định gradient ban đầu
    weight_decay=0.0005 )
    
    # LƯU Ý: YOLOv8 tự động tính toán số bước tích lũy gradient (accumulation steps) theo công thức:
    # accumulate = max(1, round(nbs / batch)) với nbs mặc định là 64.
    # Với batch=4, YOLOv8 tự động áp dụng accumulate = 64 / 4 = 16 bước để giả lập batch size 64.)

print("\n🎉 Hoàn thành chạy thử nghiệm. Hệ thống đã hoạt động bình thường!")

📥 Đang tải mô hình pre-trained YOLOv8s...
🔥 Bắt đầu chạy sanity check / trial training...
Ultralytics 8.4.75  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\Research\Yolo_Pothole_detection\Pothole_Detection\data\processed\trial_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

KeyboardInterrupt: 

## 📈 4. Script 3: Huấn Luyện Toàn Bộ Dữ Liệu (Optimized Full Training)

Khi chuyển sang huấn luyện trên **toàn bộ dữ liệu (~12,500 ảnh)**, mục tiêu của chúng ta là **tối đa hóa hiệu năng GPU (tăng GPU Utilization)** để giảm thiểu thời gian huấn luyện mà vẫn giữ VRAM nằm trong giới hạn 4GB.

### Các tối ưu để đẩy hiệu suất GPU lên cao:
1. **Tăng Batch Size (`batch=8` hoặc `batch=16`)**: Đưa batch size lên mức tối đa mà card 4GB có thể chịu được (đề xuất thử từ 8 đến 16 với imgsz 512). Điều này giúp làm đầy các nhân CUDA của GPU và loại bỏ hiện tượng "chờ đợi dữ liệu" (CPU bottleneck).
2. **Tăng Số Luồng Nạp Ảnh (`workers=4`)**: Đẩy nhanh tốc độ xử lý ảnh từ CPU để cấp dữ liệu liên tục cho GPU.
3. **Tự động cấu hình file YAML cho tập dữ liệu đầy đủ**: Tạo file `dataset.yaml` chỉ đường dẫn tới `combined_pothole`.

In [13]:
import os
from pathlib import Path
from ultralytics import YOLO

# Cấu hình đường dẫn dự án
import os
# Tự động xác định PROJECT_ROOT
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
COMBINED_DATASET_DIR = PROJECT_ROOT / 'data' / 'processed' / 'combined_pothole'
FULL_YAML_PATH = COMBINED_DATASET_DIR / 'dataset.yaml'

# 1. Đảm bảo file dataset.yaml của tập gộp đầy đủ đã tồn tại
yaml_content = f"""path: {COMBINED_DATASET_DIR.as_posix()}
train: train/images
val: val/images

names:
  0: Pothole
"""
with open(FULL_YAML_PATH, 'w', encoding='utf-8') as f:
    f.write(yaml_content)
print(f"✅ Đã xác nhận file cấu hình dataset đầy đủ tại: {FULL_YAML_PATH}")

# 2. Khởi tạo mô hình YOLOv8s
print("📥 Đang tải mô hình pre-trained YOLOv8s...")
model = YOLO('yolov8s.pt')

# 3. Huấn luyện tối ưu toàn bộ dữ liệu trên GPU RTX 3050 Ti
print("🔥 Bắt đầu huấn luyện toàn bộ dữ liệu (Full Run)...")
results = model.train(
    data=str(FULL_YAML_PATH),
    epochs=100,            # Huấn luyện đầy đủ trong 100 epochs để hội tụ tốt nhất
    
    # 🚀 TỐI ƯU HÓA HIỆU NĂNG CHO GPU 4GB VRAM
    imgsz=512,             # Giữ imgsz=512 giúp tiết kiệm VRAM rất nhiều, cho phép đẩy Batch Size lên cao
    batch=16,              # Đẩy batch lên 16 để tận dụng hết nhân CUDA của GPU rời. (Nếu bị báo lỗi Out of Memory, hãy giảm xuống 8)
    device=0,              # Huấn luyện trên GPU
    workers=4,             # Tăng số luồng nạp dữ liệu lên 4 để tăng tốc CPU nạp ảnh, giải quyết nghẽn cổ chai dataloader
    amp=True,              # Luôn bật Mixed Precision để tiết kiệm bộ nhớ và đẩy nhanh tốc độ tính toán Tensor Cores
    cache=False,           # Không sử dụng cache RAM để tránh tràn tài nguyên hệ thống
    
    # 📈 THAM SỐ HUẤN LUYỆN CHUẨN
    optimizer='AdamW',     # AdamW rất tốt cho việc học hội tụ tối ưu các đặc trưng
    lr0=0.01,              # Với batch size=16, ta có thể tự tin tăng học suất ban đầu lên 0.01
    cos_lr=True,           # Giảm học suất theo hàm Cosine
    warmup_epochs=3,       # 3 epochs warmup
    weight_decay=0.0005,
    project='Pothole_YOLOv8s', # Thư mục lưu kết quả huấn luyện
    name='full_train_opt'      # Tên của lần chạy này
)

print("🎉 Quá trình huấn luyện toàn bộ dữ liệu hoàn tất thành công!")

✅ Đã xác nhận file cấu hình dataset đầy đủ tại: d:\Research\Yolo_Pothole_detection\Pothole_Detection\data\processed\combined_pothole\dataset.yaml
📥 Đang tải mô hình pre-trained YOLOv8s...
🔥 Bắt đầu huấn luyện toàn bộ dữ liệu (Full Run)...
Ultralytics 8.4.75  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\Research\Yolo_Pothole_detection\Pothole_Detection\data\processed\combined_pothole\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_

KeyboardInterrupt: 